In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\Raw_data_1Day_2024_site_5395_Lodhi_Road_Delhi_IITM_1Day.csv")

In [3]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),...,MP-Xylene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s)
0,2024-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
1,2024-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2,2024-01-03,11.26,33.04,NaN,67.01,NaN,45.57,21.80,1.37,15.55,...,NaN,NaN,97.01,0.50,268.94,0.0,0.0,218.22,NaN,NaN
3,2024-01-04,5.30,10.70,NaN,64.80,NaN,41.05,26.65,1.37,15.35,...,NaN,NaN,97.02,0.40,275.82,0.0,0.0,218.08,NaN,NaN
4,2024-01-05,NaN,NaN,NaN,66.91,NaN,41.65,27.42,1.38,15.47,...,NaN,NaN,NaN,0.32,270.52,0.0,0.0,218.55,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,154.44,261.04,NaN,84.38,NaN,NaN,21.74,NaN,NaN,...,NaN,NaN,96.90,0.52,139.64,0.0,0.0,218.36,NaN,NaN
362,2024-12-28,80.07,141.96,NaN,81.20,NaN,NaN,23.46,NaN,NaN,...,NaN,NaN,97.02,0.60,150.36,0.0,0.0,218.52,NaN,NaN
363,2024-12-29,80.85,155.84,NaN,79.57,NaN,NaN,20.78,NaN,NaN,...,NaN,NaN,92.91,0.77,149.88,0.0,0.0,218.40,NaN,NaN
364,2024-12-30,68.41,129.22,NaN,66.07,NaN,NaN,12.97,NaN,NaN,...,NaN,NaN,91.39,0.73,144.30,0.0,0.0,218.12,NaN,NaN


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (366, 16)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['NO (µg/m³)']
Dropped rows (>70% NaN): 93
Missing values after imputation:
 Timestamp          0
PM2.5 (µg/m³)      0
PM10 (µg/m³)       0
NO2 (µg/m³)        0
NH3 (µg/m³)        0
SO2 (µg/m³)        0
CO (mg/m³)         0
Ozone (µg/m³)      0
Benzene (µg/m³)    0
RH (%)             0
WS (m/s)           0
WD (deg)           0
RF (mm)            0
TOT-RF (mm)        0
SR (W/mt2)         0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (273, 15)
    Timestamp  PM2.5 (µg/m³)  PM10 (µg/m³)  NO2 (µg/m³)  NH3 (µg/m³)  \
0  2024-01-03          11.26         33.04        67.01        38.05   
1  2024-01-04           5.30         10.70        64.80        38.05   
2  2024-01-05          70.97        130.03        66.91        38.05   
3  2024-01-06          70.97        130.03        67.19        38.05   
4  2024-01-07          70.97        130.03        65.03        38.05   

   SO2 (µg/m³)  CO (mg/m³)  Ozone (µg/m³)  Benzene (µg/m³)  RH (%)  WS (m/s)  \
0        21.80        1.32         16.115             2.66   97.01      0.50   
1        26.65        1.32         16.115             2.68   97.02      0.40   
2        27.42        1.32         16.115             2.72   69.44      0.32   
3        25.99        1.32         16.115             2.70   69.44      0.63   
4        24.76        1.32         16.115             2.68   69.44      0.51   

   WD (deg)  RF (mm)  TOT-RF (mm)  SR (W/mt2)  
0    268.94    

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO2 (µg/m³),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),Benzene (µg/m³),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2)
0,2024-01-03,-1.361248,-1.265505,-0.665294,-7.105427e-15,0.201001,0.0,-3.552714e-15,1.078085,1.664827,-0.768365,0.368212,0.0,0.0,-1.249598
1,2024-01-04,-1.480343,-1.533449,-1.030084,-7.105427e-15,1.597301,0.0,-3.552714e-15,1.095825,1.665351,-1.120436,0.520514,0.0,0.0,-1.478743
2,2024-01-05,-0.168090,-0.102218,-0.681801,-7.105427e-15,1.818981,0.0,-3.552714e-15,1.131307,0.218600,-1.402092,0.403189,0.0,0.0,-0.709469
3,2024-01-06,-0.168090,-0.102218,-0.635583,-7.105427e-15,1.407289,0.0,-3.552714e-15,1.113566,0.218600,-0.310673,0.500370,0.0,0.0,-0.693102
4,2024-01-07,-0.168090,-0.102218,-0.992119,-7.105427e-15,1.053175,0.0,-3.552714e-15,1.095825,0.218600,-0.733158,0.216796,0.0,0.0,-1.953402
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
268,2024-12-27,1.499853,1.469101,2.201849,-7.105427e-15,0.183727,0.0,-3.552714e-15,-1.281420,1.659056,-0.697951,-2.494091,0.0,0.0,-1.020453
269,2024-12-28,0.013751,0.040869,1.676949,-7.105427e-15,0.678909,0.0,-3.552714e-15,-1.281420,1.665351,-0.416294,-2.256784,0.0,0.0,-0.758572
270,2024-12-29,0.029338,0.207344,1.407896,-7.105427e-15,-0.092654,0.0,-3.552714e-15,-1.281420,1.449755,0.182226,-2.267409,0.0,0.0,-0.954982
271,2024-12-30,-0.219245,-0.111933,-0.820454,-7.105427e-15,-2.341129,0.0,-3.552714e-15,-1.281420,1.370021,0.041397,-2.390933,0.0,0.0,-1.413273


In [10]:
df.to_excel('lodhiroadIITM2024.xlsx', index=False)